In [ ]:
import os
import urllib.request
from pyspark.sql import SparkSession

os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.17.10-hotspot"

os.makedirs(r"C:\hadoop\bin", exist_ok=True)
if not os.path.exists(r"C:\hadoop\bin\winutils.exe"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/cdarlint/winutils/master/hadoop-3.3.5/bin/winutils.exe", r"C:\hadoop\bin\winutils.exe")
if not os.path.exists(r"C:\hadoop\bin\hadoop.dll"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/cdarlint/winutils/master/hadoop-3.3.5/bin/hadoop.dll", r"C:\hadoop\bin\hadoop.dll")

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ.get("PATH", "")

caminho_base = "file:///" + os.path.abspath("spark-warehouse/iceberg").replace("\\", "/")

# ==========================================
# 1. LIGANDO O MOTOR DO ICEBERG
# ==========================================
print("Baixando pacotes e iniciando o motor ICEBERG (Pode demorar uns segundos)...")
builder = SparkSession.builder.appName("Iceberg_Lab") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.3") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.meucatalogo", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.meucatalogo.type", "hadoop") \
    .config("spark.sql.catalog.meucatalogo.warehouse", caminho_base)

spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("✅ Motor ICEBERG ligado!\n")

# ==========================================
# 2. LABORATÓRIO: DDL, INSERT, UPDATE E DELETE (Via SQL)
# ==========================================
print("--- Criando Banco e Tabelas (DDL) ---")
spark.sql("CREATE NAMESPACE IF NOT EXISTS meucatalogo.faculdade")

spark.sql("""
CREATE TABLE IF NOT EXISTS meucatalogo.faculdade.notas (
    id_nota INT,
    id_aluno INT,
    disciplina STRING,
    valor_nota FLOAT
) USING iceberg
""")

print("--- 1. INSERT ---")
spark.sql("""
INSERT INTO meucatalogo.faculdade.notas VALUES
(101, 1, 'Arquitetura de Dados', 8.5),
(102, 2, 'Arquitetura de Dados', 6.0)
""")
spark.sql("SELECT * FROM meucatalogo.faculdade.notas").show()

print("--- 2. UPDATE (Mudando nota do aluno 1 para 9.5) ---")
spark.sql("""
UPDATE meucatalogo.faculdade.notas
SET valor_nota = 9.5
WHERE id_aluno = 1
""")
spark.sql("SELECT * FROM meucatalogo.faculdade.notas").show()

print("--- 3. DELETE (Removendo o aluno 2) ---")
spark.sql("""
DELETE FROM meucatalogo.faculdade.notas
WHERE id_aluno = 2
""")
spark.sql("SELECT * FROM meucatalogo.faculdade.notas").show()

Baixando pacotes e iniciando o motor ICEBERG (Pode demorar uns segundos)...
✅ Motor ICEBERG ligado!

--- Criando Banco e Tabelas (DDL) ---
--- 1. INSERT ---
+-------+--------+--------------------+----------+
|id_nota|id_aluno|          disciplina|valor_nota|
+-------+--------+--------------------+----------+
|    101|       1|Arquitetura de Dados|       8.5|
|    102|       2|Arquitetura de Dados|       6.0|
+-------+--------+--------------------+----------+

--- 2. UPDATE (Mudando nota do aluno 1 para 9.5) ---
+-------+--------+--------------------+----------+
|id_nota|id_aluno|          disciplina|valor_nota|
+-------+--------+--------------------+----------+
|    101|       1|Arquitetura de Dados|       9.5|
|    102|       2|Arquitetura de Dados|       6.0|
+-------+--------+--------------------+----------+

--- 3. DELETE (Removendo o aluno 2) ---
+-------+--------+--------------------+----------+
|id_nota|id_aluno|          disciplina|valor_nota|
+-------+--------+----------------